## Plotting numpy arrays

earthkit-plots can plot plain numpy arrays directly — no earthkit-data
wrapper required.  The trade-off is that numpy arrays carry no
geographical or meteorological metadata, so you must supply coordinates
and (optionally) metadata yourself.

In [ ]:
import earthkit.data as ekd

import earthkit.plots as ekp

### Loading data as a numpy array

We start from an ERA5 sample file and extract the values as a 2-D numpy
array.  We also pull the latitude and longitude arrays from the
underlying FieldList — these are needed to tell earthkit-plots where
each grid point sits on the globe.

In [ ]:
nc = ekd.from_source("sample", "era5-monthly-mean-2t-199312.nc")
fl = nc.to_fieldlist()

lats, lons = fl.geography.latlons()
t2m = nc.to_numpy().squeeze()

print(f"t2m shape: {t2m.shape}")
print(f"lats shape: {lats.shape}, lons shape: {lons.shape}")

### A basic plot

Pass the array as the first argument and supply `x` and `y` for
coordinates.  Without metadata, earthkit-plots cannot add an automatic
title or perform unit conversion, but you still get a fully rendered map.

In [ ]:
chart = ekp.Map(domain="Europe")
chart.contourf(t2m, x=lons, y=lats)
chart.legend()
chart.coastlines()
chart.show()

### Adding metadata

earthkit-plots accepts an optional `metadata` dict that supplies the
information normally found in a GRIB or netCDF file.  Valid keys mirror
CF-convention attributes — `units`, `long_name`, `time`, `step`, and so
on.  With metadata in place, automatic titles, unit conversion, and
auto-styles all work exactly as they do for richer source types.

In [ ]:
from datetime import datetime

metadata = {
    "units": "K",
    "long_name": "2 metre temperature",
    "time": datetime(1993, 12, 1),
}

chart = ekp.Map(domain="Europe")
chart.contourf(t2m, x=lons, y=lats, metadata=metadata, units="celsius")
chart.legend()
chart.coastlines()
chart.title("{variable_name} – {time:%B %Y}")
chart.show()

### Using `style="auto"`

Once metadata is supplied earthkit-plots can also select an automatic
style for the variable:

In [ ]:
chart = ekp.Map(domain="Europe")
chart.plot(t2m, x=lons, y=lats, metadata=metadata, units="celsius")
chart.legend()
chart.coastlines()
chart.title("{variable_name} – {time:%B %Y}")
chart.show()

### Format agnosticism

numpy arrays require a little more work than the other source types —
you need to supply coordinates explicitly and provide a `metadata` dict
if you want automatic titles and unit conversion.  But once that is in
place, the plotting calls are identical:

```python
# numpy (extra setup required)
lats, lons = fl.geography.latlons()
data = nc.to_numpy().squeeze()
metadata = {"units": "K", "long_name": "2 metre temperature", ...}

# GRIB, netCDF, xarray — no extra setup
data = ekd.from_source("sample", "era5-monthly-mean-2t-199312.grib")

# In every case, the plot call is the same:
chart = ekp.Map(domain="Europe")
chart.plot(data, units="celsius")  # (pass x=, y=, metadata= for numpy)
```

This format agnosticism is a deliberate design goal of earthkit-plots:
your visualisation code should stay as consistent as possible regardless
of where the data comes from.